# Clausius Asymmetric Quarkyonic Workflow

This notebook implements the Clausius case requested for asymmetric quarkyonic nuclear matter in `QMM`.

It scans the Clausius parameter `c` through a target incompressibility `K_0`, then computes:

- symmetric ground-state parameters `a`, `b`, `c`
- asymmetric parameters `a_n`, `a_{pn}`, `b_n`, `b_{pn}`
- consistency with the symmetric limits
- fixed-`y` asymmetric quarkyonic profiles for `y = 0.0, 0.1, ..., 0.5`
- beta-equilibrium quarkyonic EOS with leptons
- neutron-star mass-radius sequences

This notebook uses the pure `clausius` model in `QMM`, meaning:

- Clausius attractive mean field
- standard van der Waals excluded volume for the repulsive sector

If instead you want Clausius attraction with Carnahan-Starling or TVM repulsion, use the separate hybrid notebooks already added in `QMM`.


## Physics Setup

The asymmetric attractive contribution is handled in the Clausius form

$$
\varepsilon_{\mathrm{attr}}(n_p,n_n) =
-\frac{a_n(n_p^2+n_n^2)+2a_{pn}n_p n_n}{1+c n_B},
\qquad n_B=n_p+n_n.
$$

In the symmetric limit `y = 1/2`, the notebook checks that

$$
a = \frac{a_n+a_{pn}}{2}, \qquad b = \frac{b_n+b_{pn}}{2}.
$$

The constraints imposed here are:

- `E/A = -16 MeV`
- `n_0 = 0.16 fm^{-3}`
- `J = 32.5 MeV`
- `L = 58.9 MeV`
- target `K_0` values in the Clausius range


In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass


def locate_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'src' / 'qmm').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate the QMM project root from this notebook.')


ROOT = locate_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qmm.config import load_run_config
from qmm.runner import run_workflow

BASE_CONFIG_PATH = ROOT / 'examples' / 'clausius_asymmetric_base.json'
OUTPUT_ROOT = ROOT / 'results' / 'generated' / 'clausius_asymmetric_scan'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    base_config = json.load(handle)

print('ROOT =', ROOT)
print('Base config =', BASE_CONFIG_PATH)
print(json.dumps(base_config, indent=2))


## Scan Inputs

The default targets span the requested incompressibility window while keeping the runtime moderate.
Edit the next cell if you want a denser scan.


In [ ]:
# Example smaller subset:
# target_k0_values = [250.0, 280.0, 315.0]
target_k0_values = [250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 315.0]

proton_fraction_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda_momentum_mev = 300.0
branch_mode = 'target_l'
write_plots = True

print('K0 targets:', target_k0_values)
print('fixed-y values:', proton_fraction_values)
print('lambda [MeV]:', lambda_momentum_mev)
print('branch_mode:', branch_mode)


## Build One Config Per Target

The helper below starts from the executable Clausius base JSON and only overrides the run-specific pieces.


In [ ]:
def build_config_for_target_k0(target_k0: float) -> dict:
    config = deepcopy(base_config)
    run_slug = f'clausius_k0_{int(round(target_k0))}'

    config['run_name'] = run_slug
    config['model']['name'] = 'clausius'
    config['model']['parameter_search']['enabled'] = True
    config['model']['parameter_search']['target_k0'] = float(target_k0)
    config['model']['parameter_search']['parameter_min'] = 0.0
    config['model']['parameter_search']['parameter_max'] = 4.74
    config['model']['parameter_search']['scan_steps'] = 121

    config['workflows']['ground_state'] = True
    config['workflows']['quantum_critical'] = False
    config['workflows']['hadronic_eos_table'] = True
    config['workflows']['symmetric_quarkyonic'] = False
    config['workflows']['asymmetric_fit'] = True
    config['workflows']['asymmetric_quarkyonic'] = True
    config['workflows']['neutron_star'] = True

    config['quarkyonic']['lambda_momentum_mev'] = float(lambda_momentum_mev)
    config['asymmetric']['enabled'] = True
    config['asymmetric']['branch_mode'] = branch_mode
    config['asymmetric']['target_j'] = 32.5
    config['asymmetric']['target_l'] = 58.9
    config['asymmetric']['target_k0'] = float(target_k0)
    config['asymmetric']['proton_fraction_values'] = list(proton_fraction_values)
    config['asymmetric']['beta_equilibrium'] = True

    run_dir = OUTPUT_ROOT / f'K0_{int(round(target_k0))}'
    config['output']['directory'] = str(run_dir)
    config['output']['write_json'] = True
    config['output']['write_csv'] = True
    config['output']['write_plots'] = bool(write_plots)

    return config


## Run The Scan

This writes one JSON config per target `K0`, runs the Clausius workflow, and stores the returned summaries.


In [ ]:
scan_summaries = {}
failures = []
config_dir = OUTPUT_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

for target_k0 in target_k0_values:
    config_dict = build_config_for_target_k0(target_k0)
    config_path = config_dir / f'clausius_k0_{int(round(target_k0))}.json'
    config_path.write_text(json.dumps(config_dict, indent=2), encoding='utf-8')

    try:
        run_config = load_run_config(config_path)
        summary = run_workflow(run_config)
        scan_summaries[target_k0] = summary
        print(f'finished K0={target_k0:.1f} MeV -> c={summary["ground_state"]["parameter_value"]:.6f}')
    except Exception as exc:
        failures.append((target_k0, str(exc)))
        print(f'failed K0={target_k0:.1f} MeV -> {exc}')

if failures:
    print('failures detected:')
    for item in failures:
        print(item)
else:
    print('all runs completed')


## Collect Summaries

The next cell extracts the main outputs you asked for:

- `a`, `b`, `c`, `K0`
- `a_n`, `a_pn`, `b_n`, `b_pn`
- `a_pn/a_n`, `b_pn/b_n`
- neutron-star maximum mass and radius at maximum mass


In [ ]:
ground_rows = []
asym_rows = []
ns_rows = []

for target_k0, summary in sorted(scan_summaries.items()):
    ground = dict(summary['ground_state'])
    ground['target_k0_requested'] = target_k0
    ground_rows.append(ground)

    asym = dict(summary['asymmetric_fit'])
    asym['target_k0_requested'] = target_k0
    asym_rows.append(asym)

    ns = dict(summary['neutron_star'])
    ns['target_k0_requested'] = target_k0
    ns_rows.append(ns)

ground_df = pd.DataFrame(ground_rows).sort_values('target_k0_requested').reset_index(drop=True)
asym_df = pd.DataFrame(asym_rows).sort_values('target_k0_requested').reset_index(drop=True)
ns_df = pd.DataFrame(ns_rows).sort_values('target_k0_requested').reset_index(drop=True)

consistency_df = asym_df[['target_k0_requested', 'a_avg', 'b_avg', 'a_n', 'a_pn', 'b_n', 'b_pn']].copy()
consistency_df['a_from_split'] = 0.5 * (consistency_df['a_n'] + consistency_df['a_pn'])
consistency_df['b_from_split'] = 0.5 * (consistency_df['b_n'] + consistency_df['b_pn'])
consistency_df['a_avg_minus_split'] = consistency_df['a_avg'] - consistency_df['a_from_split']
consistency_df['b_avg_minus_split'] = consistency_df['b_avg'] - consistency_df['b_from_split']

ground_df.to_csv(OUTPUT_ROOT / 'ground_state_summary.csv', index=False)
asym_df.to_csv(OUTPUT_ROOT / 'asymmetric_fit_summary.csv', index=False)
ns_df.to_csv(OUTPUT_ROOT / 'neutron_star_summary.csv', index=False)
consistency_df.to_csv(OUTPUT_ROOT / 'symmetric_limit_consistency.csv', index=False)

display(ground_df[['target_k0_requested', 'parameter_value', 'a', 'b', 'K0', 'n_sat', 'binding']])
display(asym_df[['target_k0_requested', 'a_n', 'a_pn', 'b_n', 'b_pn', 'ratio_a_pn_over_a_n', 'ratio_b_pn_over_b_n', 'J', 'L']])
display(ns_df[['target_k0_requested', 'max_mass_msun', 'radius_at_max_mass_km', 'max_mass_at_upper_eos_boundary']])
display(consistency_df[['target_k0_requested', 'a_avg', 'a_from_split', 'a_avg_minus_split', 'b_avg', 'b_from_split', 'b_avg_minus_split']])


## Parameter Plots


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(ground_df['target_k0_requested'], ground_df['parameter_value'], marker='o', lw=2)
axes[0, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 0].set_ylabel('resolved c')
axes[0, 0].set_title('Clausius parameter from target K0')

axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['a'], marker='o', lw=2, label='a')
axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['b'], marker='s', lw=2, label='b')
axes[0, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 1].set_title('Symmetric nuclear-matter couplings')
axes[0, 1].legend(loc='best')

axes[1, 0].plot(asym_df['target_k0_requested'], asym_df['a_n'], marker='o', lw=2, label=r'$a_n$')
axes[1, 0].plot(asym_df['target_k0_requested'], asym_df['a_pn'], marker='s', lw=2, label=r'$a_{pn}$')
axes[1, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 0].set_ylabel(r'[MeV fm$^3$]')
axes[1, 0].set_title('Attractive asymmetric couplings')
axes[1, 0].legend(loc='best')

axes[1, 1].plot(asym_df['target_k0_requested'], asym_df['b_n'], marker='o', lw=2, label=r'$b_n$')
axes[1, 1].plot(asym_df['target_k0_requested'], asym_df['b_pn'], marker='s', lw=2, label=r'$b_{pn}$')
axes[1, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 1].set_ylabel(r'[fm$^3$]')
axes[1, 1].set_title('Repulsive asymmetric couplings')
axes[1, 1].legend(loc='best')

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(asym_df['target_k0_requested'], asym_df['ratio_a_pn_over_a_n'], marker='o', lw=2)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$a_{pn}/a_n$')
axes[0].set_title('Attractive split ratio')

axes[1].plot(asym_df['target_k0_requested'], asym_df['ratio_b_pn_over_b_n'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$b_{pn}/b_n$')
axes[1].set_title('Repulsive split ratio')

plt.show()


## Neutron-Star Summary

This summarizes the beta-equilibrium quarkyonic EOS at the neutron-star level.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

axes[0].plot(ns_df['target_k0_requested'], ns_df['max_mass_msun'], marker='o', lw=2)
axes[0].axhline(2.0, color='gray', ls='--', lw=1.0)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$M_{\max}$ [$M_\odot$]')
axes[0].set_title('Maximum neutron-star mass')

axes[1].plot(ns_df['target_k0_requested'], ns_df['radius_at_max_mass_km'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$R(M_{\max})$ [km]')
axes[1].set_title('Radius at maximum mass')

plt.show()


## Pure Hadronic EOS For One Target

The next cells inspect the pure Clausius hadronic EOS before quarkyonic matter is introduced.


In [ ]:
selected_target_k0_hadronic = target_k0_values[0]
hadronic_summary = scan_summaries[selected_target_k0_hadronic]
hadronic_csv = Path(hadronic_summary['outputs']['hadronic_eos_csv'])
hadronic_df = pd.read_csv(hadronic_csv)

print('Hadronic EOS CSV:', hadronic_csv)
display(hadronic_df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].plot(hadronic_df['n_over_n0'], hadronic_df['e_per_particle_minus_m'], lw=2.2)
axes[0].axhline(0.0, color='black', ls='--', lw=1.0)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$E/A - m_N$ [MeV]')
axes[0].set_title(f'Hadronic binding curve, K0={selected_target_k0_hadronic:.0f} MeV')

axes[1].plot(hadronic_df['n_over_n0'], hadronic_df['pressure'], lw=2.2)
axes[1].axhline(0.0, color='black', ls='--', lw=1.0)
axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1].set_title('Pure hadronic pressure')

axes[2].plot(hadronic_df['n_over_n0'], hadronic_df['vs2'], lw=2.2)
axes[2].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[2].set_ylim(-0.1, 0.8)
axes[2].set_xlabel(r'$n/n_0$')
axes[2].set_ylabel(r'$v_s^2$')
axes[2].set_title('Pure hadronic sound speed')

plt.show()


## Beta-Equilibrium Quarkyonic EOS And Mass-Radius Curve

Pick one target `K0` below and the notebook will load the beta-equilibrium EOS and neutron-star sequence from that run.


In [ ]:
selected_target_k0_beta = target_k0_values[0]
selected_summary = scan_summaries[selected_target_k0_beta]

beta_csv = Path(selected_summary['outputs']['asymmetric_beta_csv'])
ns_csv = Path(selected_summary['outputs']['neutron_star_sequence_csv'])

beta_df = pd.read_csv(beta_csv)
ns_sequence_df = pd.read_csv(ns_csv)

print('Beta-equilibrium CSV:', beta_csv)
print('Neutron-star sequence CSV:', ns_csv)
display(beta_df.head())
display(ns_sequence_df.head())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(beta_df['n_over_n0'], beta_df['y'], lw=2)
axes[0, 0].set_xlabel(r'$n/n_0$')
axes[0, 0].set_ylabel(r'$y$')
axes[0, 0].set_title('Beta-equilibrium proton fraction')

axes[0, 1].plot(beta_df['n_over_n0'], beta_df['quark_fraction'], lw=2)
axes[0, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].set_ylabel(r'$f_Q$')
axes[0, 1].set_title('Beta-equilibrium quark fraction')

beta_energy_per_baryon_gev = (beta_df['energy_density'] / beta_df['n_b'] - 938.0) / 1000.0
axes[1, 0].plot(beta_df['n_over_n0'], beta_energy_per_baryon_gev, lw=2)
axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 0].set_ylabel(r'$\epsilon/\rho_B - m_N$ [GeV]')
axes[1, 0].set_title('Beta-equilibrium energy per baryon')

axes[1, 1].plot(ns_sequence_df['radius_km'], ns_sequence_df['mass_msun'], lw=2)
axes[1, 1].set_xlabel(r'$R$ [km]')
axes[1, 1].set_ylabel(r'$M$ [$M_\odot$]')
axes[1, 1].set_title(f'Mass-radius curve, K0={selected_target_k0_beta:.0f} MeV')

plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.6), constrained_layout=True)
ax.plot(beta_df['n_over_n0'], beta_df['vs2'], lw=2)
ax.axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
selected_vs2_upper = max(1.0, 1.05 * beta_df['vs2'].max())
ax.set_ylim(-0.1, selected_vs2_upper)
ax.set_xlabel(r'$n/n_0$')
ax.set_ylabel(r'$v_s^2$')
ax.set_title(f'Beta-equilibrium sound speed, K0={selected_target_k0_beta:.0f} MeV')
plt.show()


## Beta-Equilibrium Overlay For Both Asymmetric Branches

This section overlays the beta-equilibrium Clausius results for all seven incompressibility targets in both asymmetric branches.
The color identifies `K_0`, the solid lines are for `b_{pn} \\neq b_n`, and the dashed lines are for `b_{pn}=b_n`.


In [ ]:
comparison_target_k0_values = [250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 315.0]
beta_comparison_unequal = {}
beta_comparison_equal = {}

for target_k0 in comparison_target_k0_values:
    summary = scan_summaries.get(target_k0)
    if summary is None:
        summary_path = OUTPUT_ROOT / f'K0_{int(round(target_k0))}' / f'clausius_k0_{int(round(target_k0))}_summary.json'
        if not summary_path.exists():
            raise FileNotFoundError(f'Missing summary for K0={target_k0:.1f} MeV: {summary_path}')
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    beta_csv = Path(summary['outputs']['asymmetric_beta_csv'])
    beta_comparison_unequal[target_k0] = pd.read_csv(beta_csv)

    equal_beta_csv = ROOT / 'results' / 'generated' / 'clausius_equal_b_scan_fullrange' / f'K0_{int(round(target_k0))}' / f'clausius_equal_b_k0_{int(round(target_k0))}_asymmetric_beta.csv'
    if not equal_beta_csv.exists():
        raise FileNotFoundError(f'Missing equal-b beta CSV for K0={target_k0:.1f} MeV: {equal_beta_csv}')
    beta_comparison_equal[target_k0] = pd.read_csv(equal_beta_csv)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
colors = plt.cm.viridis([index / max(len(comparison_target_k0_values) - 1, 1) for index in range(len(comparison_target_k0_values))])

for color, target_k0 in zip(colors, comparison_target_k0_values):
    df_unequal = beta_comparison_unequal[target_k0]
    df_equal = beta_comparison_equal[target_k0]
    label = f'K0={target_k0:.0f} MeV'
    energy_per_baryon_unequal_gev = (df_unequal['energy_density'] / df_unequal['n_b'] - 938.0) / 1000.0
    energy_per_baryon_equal_gev = (df_equal['energy_density'] / df_equal['n_b'] - 938.0) / 1000.0
    axes[0, 0].plot(df_unequal['n_over_n0'], df_unequal['quark_fraction'], lw=2.2, color=color, ls='-', label=label)
    axes[0, 0].plot(df_equal['n_over_n0'], df_equal['quark_fraction'], lw=2.2, color=color, ls='--')
    axes[0, 1].plot(df_unequal['n_over_n0'], df_unequal['vs2'], lw=2.2, color=color, ls='-')
    axes[0, 1].plot(df_equal['n_over_n0'], df_equal['vs2'], lw=2.2, color=color, ls='--')
    axes[1, 0].plot(df_unequal['n_over_n0'], energy_per_baryon_unequal_gev, lw=2.2, color=color, ls='-')
    axes[1, 0].plot(df_equal['n_over_n0'], energy_per_baryon_equal_gev, lw=2.2, color=color, ls='--')
    axes[1, 1].plot(df_unequal['n_over_n0'], df_unequal['pressure'], lw=2.2, color=color, ls='-')
    axes[1, 1].plot(df_equal['n_over_n0'], df_equal['pressure'], lw=2.2, color=color, ls='--')

axes[0, 0].set_xlabel(r'$n/n_0$')
axes[0, 0].set_ylabel(r'$f_Q$')
axes[0, 0].set_title('Beta-equilibrium quark fraction')
k0_legend = axes[0, 0].legend(loc='upper left', fontsize=9, title=r'$K_0$')
style_handles = [
    axes[0, 0].plot([], [], color='black', lw=2.2, ls='-', label=r'$b_{pn} \\neq b_n$')[0],
    axes[0, 0].plot([], [], color='black', lw=2.2, ls='--', label=r'$b_{pn} = b_n$')[0],
]
style_legend = axes[0, 0].legend(handles=style_handles, loc='lower right', fontsize=9, title='Branch')
axes[0, 0].add_artist(k0_legend)

axes[0, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].set_ylabel(r'$v_s^2$')
axes[0, 1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
overlay_vs2_upper = max(
    1.0,
    1.05 * max(df['vs2'].max() for df in beta_comparison_unequal.values()),
    1.05 * max(df['vs2'].max() for df in beta_comparison_equal.values()),
)
axes[0, 1].set_ylim(-0.1, overlay_vs2_upper)
axes[0, 1].set_title('Beta-equilibrium sound speed')

axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 0].set_ylabel(r'$\epsilon/\rho_B - m_N$ [GeV]')
axes[1, 0].set_title('Beta-equilibrium energy per baryon')

axes[1, 1].set_xlabel(r'$n/n_0$')
axes[1, 1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1, 1].set_title('Beta-equilibrium pressure')

comparison_png = OUTPUT_ROOT / "clausius_branch_comparison_k0_overlay.png"
comparison_pdf = OUTPUT_ROOT / "clausius_branch_comparison_k0_overlay.pdf"
fig.savefig(comparison_png, dpi=200, bbox_inches="tight")
fig.savefig(comparison_pdf, bbox_inches="tight")
plt.show()

print('saved:', comparison_png)
print('saved:', comparison_pdf)


## Fixed-`y` Asymmetric Quarkyonic Profiles

These are the non-beta-equilibrated branches for `y = 0.0, 0.1, ..., 0.5`.


In [ ]:
selected_target_k0_fixed_y = target_k0_values[0]
selected_summary_fixed_y = scan_summaries[selected_target_k0_fixed_y]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for y_value in proton_fraction_values:
    key = f'asymmetric_y_{y_value:0.3f}_csv'
    csv_path = Path(selected_summary_fixed_y['outputs'][key])
    df = pd.read_csv(csv_path)
    label = f'y={y_value:.1f}'
    axes[0].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)
    axes[1].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)

axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$f_Q$')
axes[0].set_title(f'Quark fraction for K0={selected_target_k0_fixed_y:.0f} MeV')
axes[0].legend(loc='upper left', ncol=2)

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$v_s^2$')
axes[1].set_ylim(-0.1, 0.8)
axes[1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[1].set_title(f'Sound speed for K0={selected_target_k0_fixed_y:.0f} MeV')
axes[1].legend(loc='upper right', ncol=2)

plt.show()


## Output Check

This final cell lists the generated JSON inputs and output folders for the scan.


In [ ]:
print('Generated config files:')
for path in sorted((OUTPUT_ROOT / 'configs').glob('*.json')):
    print(' -', path.relative_to(ROOT))

print('Output folders:')
for path in sorted(OUTPUT_ROOT.glob('K0_*')):
    if path.is_dir():
        print(' -', path.relative_to(ROOT))
